# 🫁 Lumora VLM — Full Local Training Pipeline

This notebook unifies all training, fine-tuning, evaluation and inference logic from:
- `train.ipynb` (local Phase 1 warmup)
- `mimic-cxr.ipynb` (Kaggle Phase 1 multi-epoch + Phase 2 joint fine-tuning)
- `evaluate_vlm.py` (local inference / report generation)

**Designed to run entirely on your local machine** (CPU, CUDA, or Apple Silicon MPS).

---

## 🏗️ Architecture Overview
```
[ Chest X-ray Image ]
        │
        ▼
[ DenseNet121 Encoder ]  →  1024-dim visual features
        │
        ▼
[ Linear Projector ]     →  768-dim (GPT-2 embedding space)
        │
        ▼  (prepend as visual prefix token)
[ GPT-2 Decoder ]        →  autoregressive report generation
        │
        ▼
[ Generated Clinical Report ]
```

## 📋 Two-Phase Training Strategy
| Phase | What trains | Learning Rate | Purpose |
|---|---|---|---|
| **Phase 1 Warmup** | Projector + GPT-2 only | `1e-4` | Align visual tokens to text space fast |
| **Phase 2 Joint** | All layers (DenseNet + Projector + GPT-2) | `2e-5` | Fine-tune the entire pipeline end-to-end |

---
## ⚙️ Step 0 — Configuration
**Edit only this cell.** All paths and hyperparameters are defined here.

In [4]:
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────
# 📁 DATA PATHS  (edit these to match your local directory layout)
# ─────────────────────────────────────────────────────────────────────
# Root directory containing the 'files/' folder of MIMIC-CXR images
IMG_ROOT = Path("official_data_iccv_final/")

# CSV files (they sit next to this notebook by default)
TRAIN_CSV = Path("mimic_cxr_aug_train.csv")
VALID_CSV = Path("mimic_cxr_aug_validate.csv")

# ─────────────────────────────────────────────────────────────────────
# 💾 CHECKPOINT PATHS
# ─────────────────────────────────────────────────────────────────────
# Set to None to start from scratch, or a .pt file path to resume
RESUME_CHECKPOINT = None          # e.g. Path("mimic_vlm_epoch1_checkpoint.pt")

# Where to save checkpoints produced by this run
CHECKPOINT_DIR = Path(".")
FINAL_MODEL_NAME = "mimic_vlm_phase2_fully_trained.pt"

# ─────────────────────────────────────────────────────────────────────
# 🔧 HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────
BATCH_SIZE         = 4     # Lower if you run out of RAM/VRAM (try 2 on CPU)
MAX_TEXT_LENGTH    = 128   # Max tokens per report
NUM_WORKERS        = 0     # Set to 2-4 on Linux/Mac; keep 0 on Windows

# Phase 1 — Warmup (encoder frozen)
PHASE1_EPOCHS      = 3
PHASE1_LR          = 1e-4

# Phase 2 — Joint fine-tuning (all layers)
PHASE2_EPOCHS      = 3
PHASE2_LR          = 2e-5

# Generation
MAX_GEN_TOKENS     = 64

print("✅ Configuration loaded.")
print(f"   IMG_ROOT   : {IMG_ROOT.resolve()}")
print(f"   TRAIN_CSV  : {TRAIN_CSV.resolve()}")
print(f"   VALID_CSV  : {VALID_CSV.resolve()}")

✅ Configuration loaded.
   IMG_ROOT   : /Users/md.nurealamsiddiquee/Projects/lumora/official_data_iccv_final
   TRAIN_CSV  : /Users/md.nurealamsiddiquee/Projects/lumora/mimic_cxr_aug_train.csv
   VALID_CSV  : /Users/md.nurealamsiddiquee/Projects/lumora/mimic_cxr_aug_validate.csv


---
## 📦 Step 1 — Imports & Hardware Detection

In [3]:
import os
import ast
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import GPT2LMHeadModel, AutoTokenizer
from tqdm.auto import tqdm

# ── Hardware detection: MPS (Apple Silicon) → CUDA → CPU ──────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍏 Apple Silicon MPS backend active.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"⚡ NVIDIA CUDA GPU active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("💻 No GPU found — running on CPU (training will be slow).")

print(f"   PyTorch version : {torch.__version__}")
print(f"   Active device   : {device}")

🍏 Apple Silicon MPS backend active.
   PyTorch version : 2.12.0
   Active device   : mps


---
## 🗄️ Step 2 — Dataset & DataLoaders

Each CSV row stores image paths and report texts as string-encoded lists.  
`MimicReportDataset` parses them safely with `ast.literal_eval`, loads the first image per row, and tokenizes the corresponding report.

In [5]:
class MimicReportDataset(Dataset):
    def __init__(self, csv_file, img_root_dir, tokenizer, max_length=128):
        self.df           = pd.read_csv(csv_file)
        self.img_root_dir = Path(img_root_dir)
        self.tokenizer    = tokenizer
        self.max_length   = max_length

        # Exact transform used during training on Kaggle
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_paths = ast.literal_eval(row['image'])
        reports   = ast.literal_eval(row['text'])

        img_path    = self.img_root_dir / img_paths[0]
        report_text = reports[0] if reports else "Findings: Normal study."

        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
        except Exception:
            image = torch.zeros(3, 224, 224)  # fallback for missing files

        tokens = self.tokenizer(
            report_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        input_ids      = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100  # ignore padding in loss

        return {
            'image':          image,
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'labels':         labels
        }


# ── Tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# ── Datasets ──────────────────────────────────────────────────────────
train_dataset = MimicReportDataset(TRAIN_CSV, IMG_ROOT, tokenizer, MAX_TEXT_LENGTH)
val_dataset   = MimicReportDataset(VALID_CSV, IMG_ROOT, tokenizer, MAX_TEXT_LENGTH)

# ── DataLoaders ───────────────────────────────────────────────────────
# pin_memory is only beneficial with CUDA; disable on CPU/MPS to avoid warnings
pin = device.type == "cuda"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

print(f"✅ Data pipelines ready!")
print(f"   Train samples : {len(train_dataset):,}  ({len(train_loader):,} batches)")
print(f"   Val   samples : {len(val_dataset):,}   ({len(val_loader):,} batches)")

✅ Data pipelines ready!
   Train samples : 64,586  (16,147 batches)
   Val   samples : 500   (125 batches)


### 🔍 Step 2b — Pipeline Sanity Check
Always run this before training. Catches path issues before wasting hours.

In [6]:
try:
    sample = next(iter(train_loader))
    print("✅ Data pipeline is healthy!")
    print(f"   Image tensor shape    : {sample['image'].shape}")
    print(f"   Input IDs shape       : {sample['input_ids'].shape}")
    print(f"   Attention mask shape  : {sample['attention_mask'].shape}")
    print(f"   Labels shape          : {sample['labels'].shape}")
except Exception as e:
    print(f"❌ Data pipeline error: {e}")
    print("   Check that IMG_ROOT, TRAIN_CSV and VALID_CSV paths are correct.")

✅ Data pipeline is healthy!
   Image tensor shape    : torch.Size([4, 3, 224, 224])
   Input IDs shape       : torch.Size([4, 128])
   Attention mask shape  : torch.Size([4, 128])
   Labels shape          : torch.Size([4, 128])


---
## 🧠 Step 3 — Model Architecture

The `MedicalReportGenerator` is a VLM (Vision-Language Model) that:
1. Extracts 1024-dim image features with **DenseNet121** (pre-trained on ImageNet)
2. Projects them to **768-dim** (GPT-2 embedding space) via a linear layer
3. Prepends this as a visual prefix token before the text tokens
4. Runs the concatenated embeddings through **GPT-2** for text generation

In [7]:
class MedicalReportGenerator(nn.Module):
    def __init__(self, vocab_size=50257, embed_dim=768):
        super().__init__()

        # 1. Vision Encoder — DenseNet121 pretrained on ImageNet
        self.encoder = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        num_ftrs = self.encoder.classifier.in_features  # 1024
        self.encoder.classifier = nn.Identity()         # strip classification head

        # 2. Projection Layer: 1024 → 768
        self.projector = nn.Linear(num_ftrs, embed_dim)

        # 3. Causal Language Decoder — GPT-2
        self.decoder = GPT2LMHeadModel.from_pretrained("gpt2")
        self.decoder.resize_token_embeddings(vocab_size)

    def forward(self, images, input_ids, attention_mask):
        # Step 1: Visual features → [batch, 1024]
        visual_features = self.encoder(images)

        # Step 2: Project → [batch, 1, 768]  (add sequence dim)
        visual_embeddings = self.projector(visual_features).unsqueeze(1)

        # Step 3: Text token embeddings → [batch, seq_len, 768]
        text_embeddings = self.decoder.transformer.wte(input_ids)

        # Step 4: Concatenate visual prefix + text → [batch, 1+seq_len, 768]
        inputs_embeds = torch.cat((visual_embeddings, text_embeddings), dim=1)

        # Step 5: Extend attention mask to cover the visual prefix token
        visual_mask   = torch.ones((images.size(0), 1), device=images.device)
        extended_mask = torch.cat((visual_mask, attention_mask), dim=1)

        return self.decoder(inputs_embeds=inputs_embeds,
                            attention_mask=extended_mask).logits


print("✅ MedicalReportGenerator class defined.")

✅ MedicalReportGenerator class defined.


---
## 🚀 Step 4 — Instantiate Model & Load Checkpoint (if resuming)

In [8]:
model = MedicalReportGenerator().to(device)
print(f"🧠 Model instantiated on {device}.")

# Track where we left off
resume_epoch     = 0
phase1_done      = False
saved_optimizer  = None

if RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists():
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)

    # Support both nested dict and raw state_dict formats
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        resume_epoch    = checkpoint.get("epoch", 0)
        saved_optimizer = checkpoint.get("optimizer_state_dict", None)
        phase1_done     = checkpoint.get("phase1_done", False)
        print(f"✅ Resumed from checkpoint (epoch {resume_epoch}, "
              f"phase1_done={phase1_done}).")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Loaded raw state_dict checkpoint.")
else:
    print("   Starting from scratch (no checkpoint to resume).")

model.eval()
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /Users/md.nurealamsiddiquee/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:09<00:00, 3.37MB/s]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

🧠 Model instantiated on mps.
   Starting from scratch (no checkpoint to resume).
   Trainable params: 132,180,864


---
## 🔧 Step 5 — Training Helpers

In [9]:
def set_phase(model, phase):
    """Freeze/unfreeze layers and return trainable parameters."""
    if phase == "warmup":
        print("🔥 Phase 1 — Encoder FROZEN. Training Projector + GPT-2 only.")
        for p in model.encoder.parameters():
            p.requires_grad = False
        for p in model.projector.parameters():
            p.requires_grad = True
        for p in model.decoder.parameters():
            p.requires_grad = True
    elif phase == "joint":
        print("🔓 Phase 2 — ALL layers UNFROZEN. Joint fine-tuning.")
        for p in model.parameters():
            p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable params: {trainable:,}")
    return filter(lambda p: p.requires_grad, model.parameters())


def run_epoch(model, loader, optimizer, criterion, device, phase_label, train=True):
    """One pass over the dataloader (training or validation)."""
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        bar = tqdm(loader, desc=phase_label, leave=True)
        for batch in bar:
            images         = batch['image'].to(device)
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            if train:
                optimizer.zero_grad()

            logits = model(images, input_ids, attention_mask)

            # Shift: predict next token; skip the visual prefix position (index 0)
            shift_logits = logits[:, 1:-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)),
                             shift_labels.view(-1))

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss,
                    phase1_done, path):
    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss":           train_loss,
        "val_loss":             val_loss,
        "phase1_done":          phase1_done,
    }, path)
    print(f"   💾 Checkpoint saved → {path}")


criterion = nn.CrossEntropyLoss(ignore_index=-100)
print("✅ Helpers defined.")

✅ Helpers defined.


---
## 🔥 Step 6 — Phase 1: Warmup Training (Encoder Frozen)

The DenseNet121 encoder stays frozen. Only the **Projector** and **GPT-2 decoder** are trained.  
This quickly aligns the visual token space to GPT-2's text embedding space.

> **Skip this cell** if `RESUME_CHECKPOINT` points to a file where Phase 1 was already completed (`phase1_done=True`).

In [ ]:
if phase1_done:
    print("⏭️  Phase 1 already complete (loaded from checkpoint). Skipping.")
else:
    trainable = set_phase(model, "warmup")
    optimizer = torch.optim.AdamW(trainable, lr=PHASE1_LR)

    # Restore optimizer state if we're resuming mid-phase
    if saved_optimizer is not None and resume_epoch < PHASE1_EPOCHS:
        optimizer.load_state_dict(saved_optimizer)
        print(f"   Optimizer state restored from checkpoint.")

    start = resume_epoch
    for epoch in range(start + 1, PHASE1_EPOCHS + 1):
        print(f"\n{'='*60}")
        print(f"  Phase 1 — Epoch {epoch}/{PHASE1_EPOCHS}")
        print(f"{'='*60}")

        train_loss = run_epoch(model, train_loader, optimizer, criterion,
                               device, f"Train P1 Epoch {epoch}", train=True)
        val_loss   = run_epoch(model, val_loader,   optimizer, criterion,
                               device, f"Val   P1 Epoch {epoch}", train=False)

        print(f"\n  📊 Epoch {epoch} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        ckpt_path = CHECKPOINT_DIR / f"mimic_vlm_phase1_epoch{epoch}_checkpoint.pt"
        save_checkpoint(model, optimizer, epoch, train_loss, val_loss,
                        phase1_done=False, path=ckpt_path)

    # Mark Phase 1 complete and save a clearly-named final Phase 1 file
    phase1_path = CHECKPOINT_DIR / "mimic_vlm_phase1_FINAL.pt"
    save_checkpoint(model, optimizer, PHASE1_EPOCHS, train_loss, val_loss,
                    phase1_done=True, path=phase1_path)
    phase1_done = True
    print(f"\n🏁 Phase 1 complete! Final checkpoint → {phase1_path}")

---
## 🔓 Step 7 — Phase 2: Joint Fine-Tuning (All Layers)

All parameters (DenseNet121 + Projector + GPT-2) are unfrozen.  
A very low learning rate (`2e-5`) prevents the pre-trained text decoder weights from catastrophically forgetting.

In [ ]:
trainable = set_phase(model, "joint")
optimizer = torch.optim.AdamW(trainable, lr=PHASE2_LR)

print(f"\n🌙 Starting Phase 2 Joint Fine-Tuning ({PHASE2_EPOCHS} epochs)...")

avg_train_loss = None
avg_val_loss   = None

for epoch in range(1, PHASE2_EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"  Phase 2 — Epoch {epoch}/{PHASE2_EPOCHS}")
    print(f"{'='*60}")

    avg_train_loss = run_epoch(model, train_loader, optimizer, criterion,
                               device, f"Train P2 Epoch {epoch}", train=True)
    avg_val_loss   = run_epoch(model, val_loader,   optimizer, criterion,
                               device, f"Val   P2 Epoch {epoch}", train=False)

    print(f"\n  📊 Epoch {epoch} — Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    ckpt_path = CHECKPOINT_DIR / f"mimic_vlm_phase2_epoch{epoch}_checkpoint.pt"
    save_checkpoint(model, optimizer, epoch, avg_train_loss, avg_val_loss,
                    phase1_done=True, path=ckpt_path)

print(f"\n🎉 Phase 2 complete!")

---
## 💾 Step 8 — Save Final Model

In [ ]:
final_path = CHECKPOINT_DIR / FINAL_MODEL_NAME
torch.save({
    "model_state_dict":     model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "final_train_loss":     avg_train_loss,
    "final_val_loss":       avg_val_loss,
    "phase1_done":          True,
    "config": {
        "phase1_epochs": PHASE1_EPOCHS,
        "phase2_epochs": PHASE2_EPOCHS,
        "phase1_lr":     PHASE1_LR,
        "phase2_lr":     PHASE2_LR,
        "batch_size":    BATCH_SIZE,
        "max_length":    MAX_TEXT_LENGTH,
    }
}, final_path)

print(f"✅ Final model saved → {final_path.resolve()}")
print(f"   File size: {final_path.stat().st_size / 1e9:.2f} GB")

---
## 🔍 Step 9 — Inference & Report Generation

Run this section any time (independently of training) to generate reports from X-ray images.
The `generate_report()` function uses greedy decoding — the same strategy as the backend API.

In [10]:
def generate_report(image_path, model, tokenizer, device,
                    max_len=MAX_GEN_TOKENS):
    """
    Generates a clinical radiology report for a chest X-ray image.

    Args:
        image_path : str or Path — path to a JPEG/PNG chest X-ray
        model      : trained MedicalReportGenerator
        tokenizer  : GPT-2 tokenizer
        device     : torch.device
        max_len    : maximum tokens to generate

    Returns:
        str — the generated report text
    """
    infer_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    if not Path(image_path).exists():
        return f"❌ File not found: {image_path}"

    try:
        raw = Image.open(image_path).convert('RGB')
        img_tensor = infer_transform(raw).unsqueeze(0).to(device)
    except Exception as e:
        return f"❌ Image load error: {e}"

    model.eval()
    with torch.no_grad():
        # Extract and project visual features
        visual_features   = model.encoder(img_tensor)
        visual_embeddings = model.projector(visual_features).unsqueeze(1)

        # Seed with BOS token
        generated     = torch.tensor([[tokenizer.bos_token_id]],
                                     dtype=torch.long).to(device)
        attn_mask     = torch.ones((1, 1), dtype=torch.long).to(device)

        for _ in range(max_len):
            text_emb      = model.decoder.transformer.wte(generated)
            inputs_embeds = torch.cat((visual_embeddings, text_emb), dim=1)

            # Extend mask to cover the visual prefix token
            vis_mask      = torch.ones((1, 1), dtype=torch.long).to(device)
            ext_mask      = torch.cat((vis_mask, attn_mask), dim=1)

            logits     = model.decoder(inputs_embeds=inputs_embeds,
                                       attention_mask=ext_mask).logits
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

            generated = torch.cat((generated, next_token), dim=1)
            attn_mask = torch.cat(
                (attn_mask, torch.ones((1, 1), dtype=torch.long).to(device)), dim=1
            )

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated[0], skip_special_tokens=True).strip()


print("✅ generate_report() ready.")

✅ generate_report() ready.


### 📊 Step 9b — Evaluate on Validation Samples
Compares model predictions against ground-truth doctor reports for the first N validation cases.

In [11]:
NUM_EVAL_SAMPLES = 5  # how many validation samples to compare

print(f"🔬 Evaluating {NUM_EVAL_SAMPLES} validation samples...\n" + "="*70)

for idx in range(NUM_EVAL_SAMPLES):
    row         = val_dataset.df.iloc[idx]
    img_paths   = ast.literal_eval(row['image'])
    reports     = ast.literal_eval(row['text'])

    full_path   = IMG_ROOT / img_paths[0]
    ground_truth = reports[0] if reports else "N/A"

    print(f"\n🔬 [Sample {idx+1}]  {img_paths[0]}")
    print("-" * 60)
    print("📝 GROUND TRUTH:")
    print(ground_truth)
    print("-" * 60)
    print("🤖 MODEL PREDICTION:")
    prediction = generate_report(full_path, model, tokenizer, device)
    print(prediction)
    print("=" * 70)

🔬 Evaluating 5 validation samples...

🔬 [Sample 1]  files/p10/p10003502/s50084553/70d7e600-373c1311-929f5ff9-23ee3621-ff551ff9.jpg
------------------------------------------------------------
📝 GROUND TRUTH:
Findings:  Impression: Compared to chest radiographs since ___, most recently ___.  Large right and moderate left pleural effusions and severe bibasilar atelectasis are unchanged.  Cardiac silhouette is obscured.  No pneumothorax.  Pulmonary edema is mild, obscured radiographically by overlying abnormalities.
------------------------------------------------------------
🤖 MODEL PREDICTION:
The first time I saw the new "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New "The New

🔬 [Sample 2]  files/p10/p10013502/s54857277/5c531aa1-a70cc60a-7cc96a81-931ae3dd-f13b5158.jpg
------------------------------------------------------------
📝 GROUND TRUTH:
Findings:  Impression: 
-----

### 🖼️ Step 9c — Generate Report for a Custom Image
Point this at any chest X-ray on disk.

In [ ]:
# ── Change this path to any chest X-ray you want to test ──────────────
TEST_IMAGE_PATH = "path/to/your/chest_xray.jpg"

print(f"🖼️  Image: {TEST_IMAGE_PATH}")
print("-" * 60)
report = generate_report(TEST_IMAGE_PATH, model, tokenizer, device)
print("🤖 Generated Report:")
print(report)

---
## ♻️ Step 10 — Load a Pre-Existing Model for Inference Only
Use this cell if you already have a trained `.pt` file and just want to run inference — no training needed.

In [12]:
INFERENCE_MODEL_PATH = Path("mimic_vlm_phase2_fully_trained.pt")

# Re-instantiate a clean model
infer_model = MedicalReportGenerator().to(device)

if INFERENCE_MODEL_PATH.exists():
    ckpt = torch.load(INFERENCE_MODEL_PATH, map_location=device)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    infer_model.load_state_dict(state)
    infer_model.eval()
    print(f"✅ Loaded weights from {INFERENCE_MODEL_PATH}")
else:
    print(f"⚠️  File not found: {INFERENCE_MODEL_PATH}")
    print("   Make sure the .pt file is in this directory, then re-run this cell.")

# Now generate a report:
# report = generate_report("your_image.jpg", infer_model, tokenizer, device)
# print(report)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Loaded weights from mimic_vlm_phase2_fully_trained.pt
